# Exercice 2 - Q-Learning avec FrozenLake-v1

Objectif : implémenter un premier algorithme de Q-Learning de zéro avec Gymnasium et NumPy.

## Étape 1 - Créer l'environnement et initialiser la Q-table

`FrozenLake-v1` possède un nombre fini d'états et d'actions. On peut donc représenter ce que l'agent apprend dans une Q-table.

In [1]:
import gymnasium as gym
import numpy as np

env = gym.make("FrozenLake-v1", is_slippery=True)

number_of_states = env.observation_space.n
number_of_actions = env.action_space.n

q_table = np.zeros((number_of_states, number_of_actions))

print("Nombre d'états :", number_of_states)
print("Nombre d'actions :", number_of_actions)
print("Dimensions de la Q-table :", q_table.shape)
print(q_table)

Nombre d'états : 16
Nombre d'actions : 4
Dimensions de la Q-table : (16, 4)
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]


## Étape 2 - Entraîner l'agent avec le Q-Learning

L'agent utilise une stratégie epsilon-greedy : il explore souvent au début, puis exploite progressivement les valeurs apprises dans la Q-table.

In [2]:
training_episodes = 50_000
max_steps_per_episode = 100

learning_rate = 0.1
discount_factor = 0.99

epsilon = 1.0
min_epsilon = 0.01
epsilon_decay = 0.0001

rng = np.random.default_rng(seed=42)

for episode in range(training_episodes):
    state, info = env.reset()
    terminated = False
    truncated = False

    for step in range(max_steps_per_episode):
        if rng.random() < epsilon:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(q_table[state, :]))

        new_state, reward, terminated, truncated, info = env.step(action)

        old_value = q_table[state, action]
        max_future_q = np.max(q_table[new_state, :])
        new_value = old_value + learning_rate * (
            reward + discount_factor * max_future_q - old_value
        )
        q_table[state, action] = new_value

        state = new_state

        if terminated or truncated:
            break

    epsilon = min_epsilon + (1.0 - min_epsilon) * np.exp(-epsilon_decay * episode)

print("Q-table entraînée :")
print(q_table)

Q-table entraînée :
[[0.51013481 0.50631154 0.50642127 0.50872658]
 [0.37490146 0.30529679 0.3714933  0.48384906]
 [0.38550233 0.40634725 0.41835881 0.47093738]
 [0.23672718 0.30658084 0.37475165 0.46074163]
 [0.52304839 0.4010411  0.23789929 0.39935923]
 [0.         0.         0.         0.        ]
 [0.28902096 0.12251992 0.19878543 0.19839372]
 [0.         0.         0.         0.        ]
 [0.3264235  0.51212881 0.26166232 0.56261772]
 [0.45511099 0.59375129 0.52783918 0.38157993]
 [0.54106999 0.41778949 0.39338088 0.36598062]
 [0.         0.         0.         0.        ]
 [0.         0.         0.         0.        ]
 [0.44474621 0.50733515 0.73331328 0.46858333]
 [0.72093279 0.85707171 0.74753724 0.77280221]
 [0.         0.         0.         0.        ]]


## Étape 3 - Évaluer l'agent entraîné

Pendant l'évaluation, l'agent n'explore plus. Il choisit toujours l'action avec la meilleure valeur dans la Q-table.

In [9]:
evaluation_episodes = 100
total_wins = 0

for episode in range(evaluation_episodes):
    state, info = env.reset()
    terminated = False
    truncated = False

    for step in range(max_steps_per_episode):
        action = int(np.argmax(q_table[state, :]))
        state, reward, terminated, truncated, info = env.step(action)

        if terminated or truncated:
            if reward == 1.0:
                total_wins += 1
            break

success_rate = total_wins / evaluation_episodes * 100
print(f"Taux de réussite sur {evaluation_episodes} épisodes : {success_rate:.0f}%")

env.close()

Taux de réussite sur 100 épisodes : 76%
